# RandomForest Cirrhosis Optimized
Target: Cirrhosis_Status

In [13]:
!pip install -q xgboost

In [46]:
# ==========================================================
# Liver Cirrhosis Prediction using LightGBM
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier



# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("/content/drive/MyDrive/cirrhosis.csv")

df.columns = df.columns.str.strip()


# Remove missing target

df = df.dropna(subset=["Stage"])


print("="*60)
print("Dataset Shape :", df.shape)
print("="*60)



# ==========================================================
# Features / Target
# ==========================================================

X = df.drop(
    columns=["ID","Stage"],
    errors="ignore"
)


y = df["Stage"].astype(int)



print("\nClasses:")
print(y.unique())



# ==========================================================
# Feature Types
# ==========================================================

num_cols = X.select_dtypes(
    include=["int64","float64"]
).columns


cat_cols = X.select_dtypes(
    include=["object"]
).columns



print("\nNumerical Features :",len(num_cols))
print("Categorical Features :",len(cat_cols))



# ==========================================================
# Preprocessing
# ==========================================================

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )

])



categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )

])



preprocessor = ColumnTransformer([

    (
        "num",
        numeric_pipeline,
        num_cols
    ),

    (
        "cat",
        categorical_pipeline,
        cat_cols
    )

])



# ==========================================================
# Train/Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)



# ==========================================================
# LightGBM Model
# ==========================================================

lgbm = LGBMClassifier(

    n_estimators=500,

    learning_rate=0.03,

    max_depth=6,

    num_leaves=31,

    min_child_samples=10,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=0.1,

    reg_lambda=0.1,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1

)



# ==========================================================
# Pipeline
# ==========================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "lightgbm",
        lgbm
    )

])



# ==========================================================
# Training
# ==========================================================

print("\nTraining LightGBM...\n")


model.fit(

    X_train,

    y_train

)



# ==========================================================
# Prediction
# ==========================================================

train_pred = model.predict(X_train)

test_pred = model.predict(X_test)



# ==========================================================
# Evaluation
# ==========================================================

print("="*60)


print(
    "Train Accuracy :",
    round(
        accuracy_score(
            y_train,
            train_pred
        )*100,
        2
    ),
    "%"
)



print(
    "Test Accuracy :",
    round(
        accuracy_score(
            y_test,
            test_pred
        )*100,
        2
    ),
    "%"
)


print("="*60)



print("\nClassification Report\n")


print(
    classification_report(
        y_test,
        test_pred
    )
)



print("\nConfusion Matrix\n")


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)



# ==========================================================
# Cross Validation
# ==========================================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)



scores = cross_val_score(

    model,

    X,

    y,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)



print("\nCross Validation Scores")

print(scores)



print(
    "\nMean Accuracy:",
    round(scores.mean()*100,2),
    "%"
)



print(
    "STD:",
    round(scores.std()*100,2),
    "%"
)

Dataset Shape : (412, 20)

Classes:
[4 3 2 1]

Numerical Features : 11
Categorical Features : 7

Training LightGBM...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000084 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 873
[LightGBM] [Info] Number of data points in the train set: 329, number of used features: 27
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train Accuracy : 100.0 %
Test Accuracy : 40.96 %

Classification Report

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         4
           2       0.24      0.32      0.27        19
           3       0.33      0.39      0.36        31
           4       0.76      0.55      0.64        29

    accuracy                           0.41        83
   macro avg       0.33      0.31      0.32        83
weighted avg       0.45      0.41      0.42        83


Confusion Matrix

[[ 0  3  1  0]
 [ 1  6 12  0]
 [ 0 14 12  5]
 [ 0  2 11 16]]


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Cross Validation Scores
[0.44578313 0.53012048 0.52439024 0.6097561  0.45121951]

Mean Accuracy: 51.23 %
STD: 6.02 %


In [45]:
!pip install lightgbm